#### Prompt Preparation

In [1]:
def default_params(): 
    return {
        'dataset': {
            'path': '/workspaces/CodeSmells/semeru-datasets/code_smells/pipeline/curated',
            'content_column': 'code',
            'sampling_size': 500,
            'prompt_column': 'prompt',
        },
        'prompts' : {
            'P1' : lambda code: 
            f"""You are an expert software engineer who writes clean, maintainable, and production-quality code. 
            Your goal is to complete the provided code without introducing any unnecessary imports.
            Specifically, do not include any import statements unless the imported module or name is actually used in the code.

            Complete the following code:\n{code}
            """,
        },
        'output_path' : '/workspaces/CodeSmells/data/extension/mitigation/datasets',
        's_msg_id': 'W0611', ### SMELL FOR CASE STUDY
    }
params = default_params()

### Imports

In [2]:
import pandas as pd
import os

In [3]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

### Dataset Loading

In [4]:
df_dataset = pd.read_json(f"{params['dataset']['path']}_{params['dataset']['sampling_size']}.json")

### Case Study Dataset Construction

In [5]:
def create_prompt_dataset(prompt_id, df_dataset):
    df_prompt = df_dataset.copy()
    df_prompt['prompt_id'] = prompt_id
    df_prompt[params['dataset']['prompt_column']] = params['prompts'][prompt_id]('')
    df_prompt['original_code'] = df_prompt[params['dataset']['content_column']]
    df_prompt[params['dataset']['content_column']] = df_prompt['original_code'].apply(params['prompts'][prompt_id])
    return df_prompt

In [6]:
df_dataset_cs = df_dataset[df_dataset['s_msg_id'] == params['s_msg_id']].copy()
df_dataset_prompted = create_prompt_dataset('P1', df_dataset_cs)

#### Store the datasets

In [7]:
output_path = params['output_path']
create_folder(output_path)
df_dataset_cs.to_json(f"{output_path}/base.json")
df_dataset_prompted.to_json(f"{output_path}/prompted.json")